# Notebook 11: Which transcription factors track the coupling?

**Goal.** Estimate transcription-factor activity from the expression of each factor's target genes, then ask which factor explains differences in MLCI between mice (H5, regulatory arm).

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** **get_net** loads a table of 'transcription factor -> target gene' relations. If you installed *decoupler* (pip install decoupler==1.8.0) it downloads CollecTRI, a curated network of hundreds of factors; otherwise it uses a built-in mini-network of the TFEB 'CLEAR' lysosomal-biogenesis targets so the notebook still runs.

*You should see:* Either many TFs (CollecTRI) or a single TFEB entry with 21 targets (built-in).

In [ ]:
from mlc.dataset import Dataset
from mlc.analysis import get_net, tf_activity
from mlc.stats import ols, design, bh_fdr
from mlc import plots
sheet = pd.read_csv(SHEET).set_index("sample_id")
net = get_net("mouse")
print(net.groupby("source").size().sort_values(ascending=False).head(5))

**Cell 3.** Per cell, the activity of a factor = the average (z-scored) expression of its target genes. Then averaged per mouse, so each mouse has one activity per factor next to its MLCI. **Caution:** activity is inferred from target expression, not measured; TFEB is controlled mostly by moving into the nucleus, which RNA data cannot see.

*You should see:* (19, ...) = 19 mice.

In [ ]:
rows = []
for c in cohorts()["mouse"]:
    ds = Dataset.load(PROC / f"{c}_scored")
    act = tf_activity(ds, net); act["sample_id"] = ds.obs.sample_id.values
    per = act.groupby("sample_id").mean(); per["cohort"] = c; rows.append(per)
TF = pd.concat(rows)
TF["log2_age"] = np.log2(sheet.loc[TF.index, "age_months"].astype(float).values)
R = pd.read_csv(RES / "mlci_per_sample.csv")
TF["mlci"] = R[R.level == "M2"].set_index("sample_id").z.reindex(TF.index)
TF = TF.dropna(subset=["mlci"]); print(TF.shape, "mice")

**Cell 4.** For each factor, two small regressions across mice: (1) **MLCI ~ factor activity + age + cohort** (does activity explain coupling beyond age?) and (2) **activity ~ age + cohort** (does activity itself change with age?). **beta_mlci** positive = higher activity goes with stronger coupling. With hundreds of factors, use **fdr_mlci**, not the raw p.

*You should see:* TFEB: beta_mlci about +0.09, p about 0.005; its age effect is negative but not significant (p about 0.2).

In [ ]:
tfs = [c for c in TF.columns if c not in ("cohort", "log2_age", "mlci")]; out = []
for t in tfs:
    d = TF.assign(tfv=TF[t]); X, nm = design(d, ["tfv", "log2_age"], cohort="cohort"); f = ols(d.mlci, X, nm)
    Xa, na = design(d, ["log2_age"], cohort="cohort"); fa = ols(d[t], Xa, na)
    out.append(dict(tf=t, beta_mlci=f.loc["tfv", "beta"], p_mlci=f.loc["tfv", "p"],
                    beta_age=fa.loc["log2_age", "beta"], p_age=fa.loc["log2_age", "p"]))
tfs_df = pd.DataFrame(out); tfs_df["fdr_mlci"] = bh_fdr(tfs_df.p_mlci.values)
tfs_df.sort_values("p_mlci").to_csv(RES / "tf_vs_mlci.csv", index=False)
print(tfs_df.sort_values("p_mlci").head(8).round(4))

**Cell 5.** (a) Strength of association with MLCI for the top factors. (b) The best candidate, one point per mouse, coloured by cohort. If the points rise from left to right within each cohort, mice with higher regulon activity keep tighter coupling.

In [ ]:
plots.fig_tf(TF, tfs_df); plt.show()